In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from datasets import Dataset, concatenate_datasets
import random
import pandas as pd
from datasets import DatasetDict
random.seed(12)
import os
import shutil
from datasets import load_from_disk
from collections import Counter

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content

!git clone https://github.com/DarynaKalinchuk/infl_b

%cd infl_b
!git pull origin main

/content
Cloning into 'infl_b'...
remote: Enumerating objects: 927, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 927 (delta 116), reused 126 (delta 66), pack-reused 740 (from 1)
Receiving objects: 100% (927/927), 19.82 MiB | 22.16 MiB/s, done.
Resolving deltas: 100% (507/507), done.
/content/infl_b
From https://github.com/DarynaKalinchuk/infl_b
 * branch            main       -> FETCH_HEAD
Already up to date.


# Backdoor

(from https://github.com/plumprc/Failures-of-Influence-Functions-in-LLMs)

In [ ]:
backdoor = load_from_disk("datasets/Backdoor")

# SafetyHarm

harmful samples

In [ ]:
ds_harm = load_dataset("PKU-Alignment/PKU-SafeRLHF")

In [ ]:
ds_harm

DatasetDict({
    train: Dataset({
        features: ['prompt', 'response_0', 'response_1', 'prompt_source', 'response_0_source', 'response_1_source', 'is_response_0_safe', 'is_response_1_safe', 'response_0_harm_category', 'response_1_harm_category', 'response_0_severity_level', 'response_1_severity_level', 'better_response_id', 'safer_response_id', 'response_0_sha256', 'response_1_sha256'],
        num_rows: 73907
    })
    test: Dataset({
        features: ['prompt', 'response_0', 'response_1', 'prompt_source', 'response_0_source', 'response_1_source', 'is_response_0_safe', 'is_response_1_safe', 'response_0_harm_category', 'response_1_harm_category', 'response_0_severity_level', 'response_1_severity_level', 'better_response_id', 'safer_response_id', 'response_0_sha256', 'response_1_sha256'],
        num_rows: 8211
    })
})

In [ ]:
ds = ds_harm["train"]

KEEP_CATEGORIES = [
    "Privacy Violation",
    "Copyright Issues",
    "Cybercrime",
    "Economic Crime",
    "Violence",
    "Insulting Behavior",
    "Mental Manipulation",
    "Discriminatory Behavior",
    "Disrupting Public Order",
    "Environmental Damage",
]

def extract_category(category_dict):
    for category, value in category_dict.items():
        if value:
            return category
    return "safe"

def sample_len(x):
    return len(x["prompt"]) + len(x["response_0"])

ds = ds.map(
    lambda x: {
        "category": extract_category(x["response_0_harm_category"]),
        "sample_len": sample_len(x),
    }
)

harmful = ds.filter(
    lambda x: (
        not x["is_response_0_safe"]
        and x["category"] in KEEP_CATEGORIES
    )
)

train_harmful_parts = []
test_harmful_parts = []

for cat in KEEP_CATEGORIES:
    cat_ds = (
        harmful
        .filter(lambda x, cat=cat: x["category"] == cat)
        .sort("sample_len")  # shortest first
    )

    train_harmful_parts.append(cat_ds.select(range(30)))
    test_harmful_parts.append(cat_ds.select(range(30, 50)))

train = Dataset.from_dict({
    "category": sum([list(part["category"]) for part in train_harmful_parts], []),
    "prompts": sum([list(part["prompt"]) for part in train_harmful_parts], []),
    "response": sum([list(part["response_0"]) for part in train_harmful_parts], []),
}).shuffle(seed=42)

test = Dataset.from_dict({
    "category": sum([list(part["category"]) for part in test_harmful_parts], []),
    "prompts": sum([list(part["prompt"]) for part in test_harmful_parts], []),
    "response": sum([list(part["response_0"]) for part in test_harmful_parts], []),
}).shuffle(seed=42)

print(train)
print(test)

print("\nTrain harmful categories:")
print(Counter(train["category"]))

print("\nTest harmful categories:")
print(Counter(test["category"]))

Dataset({
    features: ['category', 'prompts', 'response'],
    num_rows: 300
})
Dataset({
    features: ['category', 'prompts', 'response'],
    num_rows: 200
})

Train harmful categories:
Counter({'Disrupting Public Order': 30, 'Environmental Damage': 30, 'Cybercrime': 30, 'Insulting Behavior': 30, 'Privacy Violation': 30, 'Violence': 30, 'Mental Manipulation': 30, 'Discriminatory Behavior': 30, 'Copyright Issues': 30, 'Economic Crime': 30})

Test harmful categories:
Counter({'Discriminatory Behavior': 20, 'Copyright Issues': 20, 'Insulting Behavior': 20, 'Cybercrime': 20, 'Privacy Violation': 20, 'Economic Crime': 20, 'Disrupting Public Order': 20, 'Environmental Damage': 20, 'Mental Manipulation': 20, 'Violence': 20})


In [ ]:
def word_len(example):
    return len((example["prompts"] + " " + example["response"]).split())

train_lengths = [word_len(x) for x in train]
test_lengths = [word_len(x) for x in test]

print("Train")
print("  Min words:", min(train_lengths))
print("  Max words:", max(train_lengths))

print("Test")
print("  Min words:", min(test_lengths))
print("  Max words:", max(test_lengths))

Train
  Min words: 5
  Max words: 90
Test
  Min words: 10
  Max words: 103


In [ ]:
train = train.remove_columns("category")
train = train.add_column("variation", ["harmful"] * len(train))

test = test.remove_columns("category")
test = test.add_column("variation", ["harmful"] * len(test))

Flattening the indices:   0%|          | 0/300 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
print("\nTrain harmful variations:")
print(Counter(train["variation"]))

print("\nTest harmful variations:")
print(Counter(test["variation"]))


Train harmful variations:
Counter({'harmful': 300})

Test harmful variations:
Counter({'harmful': 200})


safe samples

In [ ]:
ds_safe = load_dataset("databricks/databricks-dolly-15k")['train']

In [ ]:
ds_safe = ds_safe.filter(lambda x: x["category"] == "open_qa")

ds_safe = (
    ds_safe
    .rename_column("instruction", "prompts")
)


ds_safe = ds_safe.map(
    lambda x: {"sample_word_len": word_len(x)}
)

ds_safe = ds_safe.filter(
    lambda x: 5 <= x["sample_word_len"] <= 103
)

ds_safe = (
    ds_safe
    .shuffle(seed=4)
    .select(range(min(800, len(ds_safe))))
)

ds_safe = (
    ds_safe
    .remove_columns(["context", "category", "sample_word_len"])
)

print(ds_safe)

Map:   0%|          | 0/3742 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3742 [00:00<?, ? examples/s]

Dataset({
    features: ['prompts', 'response'],
    num_rows: 800
})


In [ ]:
ds_safe = ds_safe.add_column("variation", ["safe"] * len(ds_safe))


In [ ]:
ds_safe

Dataset({
    features: ['prompts', 'response', 'variation'],
    num_rows: 800
})

merging

In [ ]:
train = concatenate_datasets([train, ds_safe]).shuffle(seed=42)

In [ ]:
dataset_harm_complete = DatasetDict({
    "train": train,
    "test": test,
})


dataset_harm_complete

DatasetDict({
    train: Dataset({
        features: ['prompts', 'response', 'variation'],
        num_rows: 1100
    })
    test: Dataset({
        features: ['prompts', 'response', 'variation'],
        num_rows: 200
    })
})

In [ ]:
dataset_harm_complete.save_to_disk("datasets/SafetyHarm")

Saving the dataset (0/1 shards):   0%|          | 0/1100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

## Personas

In [ ]:
ds_roles = load_dataset("ChengyuDu0123/HER-Dataset", "sft_single_turn")
ds_train_single_turn = ds_roles['train'].filter(
    lambda x: x["turn_index"] == 0
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def extract_features(example):
    prompts = []
    replies = []

    for msg in example["messages"]:
        if msg["role"] == "user":
            prompts.append(msg["content"])
        elif msg["role"] == "assistant":
            replies.append(msg["content"])

    return {
        "prompts": prompts,
        "response": replies,
        "variation": example["character"],
    }

# Apply transformation
ds_processed = ds_train_single_turn.map(extract_features)

# Optional: keep only the new columns
columns_to_keep = ["prompts", "response", "variation"]

ds_processed = ds_processed.remove_columns(
    [col for col in ds_processed.column_names if col not in columns_to_keep]
)



In [ ]:
import re

def clean_response_text(text, keep_actions=True, keep_thoughts=True, quote_dialogue=False):
    text = re.sub(
        r"<system_thinking\b[^>]*>.*?</system_thinking>",
        "",
        text,
        flags=re.S | re.I
    )

    if keep_actions:
        text = re.sub(
            r"<role_action\b[^>]*>(.*?)</role_action>",
            lambda m: f" *{m.group(1).strip()}* ",
            text,
            flags=re.S | re.I
        )
    else:
        text = re.sub(
            r"<role_action\b[^>]*>.*?</role_action>",
            "",
            text,
            flags=re.S | re.I
        )

    if keep_thoughts:
        text = re.sub(
            r"<role_thinking\b[^>]*>(.*?)</role_thinking>",
            lambda m: f" ({m.group(1).strip()}) ",
            text,
            flags=re.S | re.I
        )
    else:
        text = re.sub(
            r"<role_thinking\b[^>]*>.*?</role_thinking>",
            "",
            text,
            flags=re.S | re.I
        )

    text = re.sub(r"</?[^>]+>", "", text)

    if quote_dialogue:
        text = re.sub(
            r"^([A-Z][A-Za-z .'-]+):\s*(.+)$",
            lambda m: f'{m.group(1)}: "{m.group(2).strip()}"',
            text,
            flags=re.M
        )

    text = re.sub(r"\s+", " ", text)

    text = re.sub(r"\*\s+", "*", text)
    text = re.sub(r"\s+\*", "*", text)

    text = re.sub(r"\*(?=[A-Za-z(])", "* ", text)
    text = re.sub(r"\)(?=[A-Za-z*])", ") ", text)

    text = re.sub(r"\s+([,.!?;:])", r"\1", text)

    return text.strip()


def normalize_text_field(value):
    if isinstance(value, list):
        return "\n".join(str(v).strip() for v in value if v)
    elif value is None:
        return ""
    return str(value).strip()


def clean_response(example):
    character = example.get("variation")

    prompts = normalize_text_field(example.get("prompts"))
    example["prompts"] = clean_response_text(
        prompts,
        keep_actions=False,
        keep_thoughts=False, quote_dialogue=True
    )

    response = normalize_text_field(example.get("response"))
    cleaned = clean_response_text(response, keep_actions=True, keep_thoughts=True, quote_dialogue=False)

    if character:
        cleaned = re.sub(
            rf"^{re.escape(character)}\s*:\s*",
            "",
            cleaned,
            flags=re.I
        )

    example["response"] = cleaned
    return example


ds_processed1 = ds_processed.map(clean_response)

# Dropping empty prompts
ds_processed1 = ds_processed1.filter(
    lambda x: x["prompts"] and x["prompts"].strip()
)

ds_processed1 = ds_processed1.map(
    lambda x: {
        "prompts": f"Act like {x['variation']}. {x['prompts']}"
    }
)

Map:   0%|          | 0/76883 [00:00<?, ? examples/s]

Filter:   0%|          | 0/76883 [00:00<?, ? examples/s]

Map:   0%|          | 0/52584 [00:00<?, ? examples/s]

In [ ]:
ds_processed1

Dataset({
    features: ['prompts', 'response', 'variation'],
    num_rows: 52584
})

In [ ]:
# for i in range(min(5, len(ds_processed1))):
#     sample = ds_processed1[i]

#     print(f"\n===== SAMPLE {i+1} =====")
#     print("VARIATION:")
#     print(sample.get("variation"))

#     print("\nPROMPTS:")
#     print(sample.get("prompts"))

#     print("\RESPONSE:")
#     print(sample.get("response"))

In [ ]:
from collections import Counter

counts = Counter(ds_processed1["variation"])

for variation, count in counts.most_common(30):
    print(f"{variation}: {count}")

Rose Hathaway: 334
Eragon: 314
Harry Dresden: 287
Sookie Stackhouse: 242
Narrator: 239
Saphira: 210
Harry Potter: 194
Dimitri Belikov: 188
Aelin Ashryver Whitethorn Galathynius: 188
Roland Deschain: 188
Arthur Dent: 176
Percy Jackson: 144
Emily Fields: 139
Hanna Marin: 136
Aria Montgomery: 136
Ford Prefect: 132
Susannah Dean: 129
Kvothe: 129
Annabeth Chase: 127
Rand al'Thor: 127
Chaol Westfall: 126
America Singer: 125
Thomas: 123
Eddie Dean: 116
Roran: 115
Ron Weasley: 115
Hermione Granger: 114
Arya: 112
Spencer Hastings: 109
Chloe Saunders: 108


In [ ]:
characters = [
    "Rose Hathaway",
    "Eragon",
    "Harry Dresden",
    "Sookie Stackhouse",
    "Harry Potter",
    "Hanna Marin",
    "Aelin Ashryver Whitethorn Galathynius",
    "Roland Deschain",
    "Arthur Dent",
    "Percy Jackson",
]

train_parts = []
test_parts = []


for character in characters:
    subset = (
        ds_processed1
        .filter(lambda x: x["variation"] == character)
        .map(lambda x: {"prompt_length": len(x["prompts"])})
        .sort("prompt_length")
        .select(range(125))  # shortest
        .shuffle(seed=42)
    )

    train_parts.append(subset.select(range(100)))
    test_parts.append(subset.select(range(100, 125)))

train_ds = concatenate_datasets(train_parts).shuffle(seed=42)
test_ds = concatenate_datasets(test_parts).shuffle(seed=42)



Filter:   0%|          | 0/52584 [00:00<?, ? examples/s]

Map:   0%|          | 0/334 [00:00<?, ? examples/s]

Filter:   0%|          | 0/52584 [00:00<?, ? examples/s]

Map:   0%|          | 0/314 [00:00<?, ? examples/s]

Filter:   0%|          | 0/52584 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/188 [00:00<?, ? examples/s]

Filter:   0%|          | 0/52584 [00:00<?, ? examples/s]

Map:   0%|          | 0/176 [00:00<?, ? examples/s]

Filter:   0%|          | 0/52584 [00:00<?, ? examples/s]

Map:   0%|          | 0/144 [00:00<?, ? examples/s]

In [ ]:
# # longest
# for ds_name, ds in [("Train", train_ds), ("Test", test_ds)]:
#     longest_idx = max(
#         range(len(ds)),
#         key=lambda i: len(ds[i]["prompts"]) + len(ds[i]["response"])
#     )

#     sample = ds[longest_idx]

#     print(f"\n{'='*50}")
#     print(ds_name)
#     print("=" * 50)
#     print("Character:", sample["variation"])
#     print("Prompt length:", len(sample["prompts"]))
#     print("Response length:", len(sample["response"]))
#     print("Total length:", len(sample["prompts"]) + len(sample["response"]))
#     print("\nPrompt:\n", sample["prompts"])
#     print("\nResponse:\n", sample["response"])

In [ ]:
train_ds = train_ds.remove_columns("prompt_length")
test_ds = test_ds.remove_columns("prompt_length")

print(train_ds)
print(test_ds)

dataset_role = DatasetDict({
    "train": train_ds,
    "test": test_ds,
})

dataset_role.save_to_disk("datasets/Personas")

Dataset({
    features: ['prompts', 'response', 'variation'],
    num_rows: 1000
})
Dataset({
    features: ['prompts', 'response', 'variation'],
    num_rows: 250
})


Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/250 [00:00<?, ? examples/s]